# 06 — OpenAI CLIP ViT-L/14@336 Inference (Round 3 of the iterative ensemble)

CLIP cosine score gallery × query. This is Round 3 in the 4-round ensemble (UIT → BLIP-2 → **CLIP** → PE-G14).

**Model:** `openai/clip-vit-large-patch14-336` (paper-faithful — OpenAI weights, not EVA-CLIP).

**A100-80GB tuning:** batch image 256, batch text 512, BF16 autocast.

**Outputs:**
- `features/clip/{gallery,queries,val,val_zs}.npz`
- `features/clip/scores_clip.pt` + val variants

**ETA:** ~30 min on A100 80GB BF16.

In [ ]:
from pathlib import Path
import os, sys, subprocess, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

NOTEBOOK_DIR = Path('.').resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from aic_colab_utils import (
    setup_aic2026_environment, select_a100_device,
    sync_chunk_to_drive, wait_for_pending_syncs,
    save_npz_atomic,
)

PATHS = setup_aic2026_environment()
INPUT_ROOT = PATHS['input_root']; MANIFEST_DIR = PATHS['manifest_dir']
LOCAL_ROOT = PATHS['local_root']; OUTPUT_ROOT = PATHS['output_root']
DRIVE_OUTPUT_ROOT = PATHS['drive_output_root']

CLIP_FEAT_DIR = OUTPUT_ROOT / 'features' / 'clip'
CLIP_FEAT_DIR.mkdir(parents=True, exist_ok=True)
device = select_a100_device()

In [ ]:
def pip_install(*p):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *p])
try:
    from transformers import CLIPModel, CLIPProcessor  # noqa
except Exception:
    pip_install('transformers>=4.40', 'safetensors')
    from transformers import CLIPModel, CLIPProcessor

import torch, torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader

MODEL_ID = 'openai/clip-vit-large-patch14-336'
processor = CLIPProcessor.from_pretrained(MODEL_ID)
model = CLIPModel.from_pretrained(MODEL_ID).to(device).eval()
model = model.to(memory_format=torch.channels_last)
amp_dtype = torch.bfloat16
print('CLIP loaded:', MODEL_ID)

In [ ]:
gallery_df = pd.read_parquet(MANIFEST_DIR / 'gallery_manifest.parquet')
query_df = pd.read_parquet(MANIFEST_DIR / 'query_manifest.parquet')
val_df = pd.read_parquet(MANIFEST_DIR / 'val_split.parquet')
val_zs_df = pd.read_parquet(MANIFEST_DIR / 'val_zeroshot_scene.parquet')

def remap(p):
    if not isinstance(p, str) or not p: return p
    if p.startswith(str(INPUT_ROOT)): return p
    for a in ('annotation/', 'train/imgs_', 'name-masked_test-set/', 'gallery/'):
        i = p.find(a)
        if i >= 0: return str(INPUT_ROOT / p[i:])
    return p
for d in (gallery_df, val_df, val_zs_df):
    if 'image_path' in d.columns:
        d['image_path'] = d['image_path'].map(remap)
print('gallery:', len(gallery_df), 'queries:', len(query_df))

IMG_BATCH = 256
TXT_BATCH = 512
NUM_WORKERS = 12

class _ImageDS(Dataset):
    def __init__(self, df, id_col):
        self.paths = df['image_path'].astype(str).tolist()
        self.ids = df[id_col].astype(str).tolist()
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        try:
            pil = Image.open(self.paths[i]).convert('RGB'); ok = True
        except Exception:
            pil = Image.new('RGB', (336, 336)); ok = False
        proc = processor(images=pil, return_tensors='pt')
        return proc['pixel_values'][0], self.ids[i], self.paths[i], ok

In [ ]:
@torch.inference_mode()
def encode_images(df, id_col):
    ids, paths, embs, ok_arr = [], [], [], []
    loader = DataLoader(_ImageDS(df, id_col), batch_size=IMG_BATCH,
                        num_workers=NUM_WORKERS, pin_memory=True,
                        persistent_workers=(NUM_WORKERS > 0), prefetch_factor=4)
    for px, b_ids, b_paths, b_ok in tqdm(loader, desc=f'clip-img {id_col}'):
        px = px.to(device, non_blocking=True).contiguous(memory_format=torch.channels_last)
        with torch.autocast(device_type=device.type, dtype=amp_dtype):
            f = model.get_image_features(pixel_values=px)
        f = F.normalize(f.float(), dim=-1).cpu().numpy().astype('float16')
        embs.append(f); ids.extend(b_ids); paths.extend(b_paths)
        ok_arr.extend([bool(x) for x in b_ok])
    return np.concatenate(embs, 0), np.array(ids), np.array(paths), np.array(ok_arr)

@torch.inference_mode()
def encode_texts(ids, texts):
    embs = []
    for s in tqdm(range(0, len(texts), TXT_BATCH), desc='clip-txt'):
        batch = list(texts[s:s + TXT_BATCH])
        tok = processor(text=batch, padding=True, truncation=True, max_length=77, return_tensors='pt').to(device)
        with torch.autocast(device_type=device.type, dtype=amp_dtype):
            f = model.get_text_features(**tok)
        embs.append(F.normalize(f.float(), dim=-1).cpu().numpy().astype('float16'))
    return np.concatenate(embs, 0), np.array([str(x) for x in ids])

g_emb, g_ids, g_paths, g_ok = encode_images(gallery_df, 'gallery_id')
q_emb, q_ids = encode_texts(query_df['query_index'].astype(str).tolist(), query_df['caption'].astype(str).tolist())
save_npz_atomic(CLIP_FEAT_DIR / 'gallery.npz', ids=g_ids, paths=g_paths, embeddings=g_emb, ok=g_ok)
save_npz_atomic(CLIP_FEAT_DIR / 'queries.npz', ids=q_ids, embeddings=q_emb)

for name, df in [('val', val_df), ('val_zs', val_zs_df)]:
    if len(df) == 0: continue
    v_img, vi_ids, vp, vok = encode_images(df, 'image_id')
    v_txt, _ = encode_texts(df['image_id'].astype(str).tolist(), df['caption'].astype(str).tolist())
    save_npz_atomic(CLIP_FEAT_DIR / f'{name}.npz', ids=vi_ids, paths=vp, embeddings=v_img, ok=vok)
    save_npz_atomic(CLIP_FEAT_DIR / f'{name}_text.npz', ids=vi_ids, embeddings=v_txt)

print('Embeddings saved.')

In [ ]:
def _score(q_emb, g_emb):
    Q = F.normalize(torch.from_numpy(q_emb.astype('float32')).to(device), dim=-1)
    G = F.normalize(torch.from_numpy(g_emb.astype('float32')).to(device), dim=-1)
    return (Q @ G.T).half().cpu()

S = _score(q_emb, g_emb)
torch.save({'scores': S, 'query_ids': q_ids, 'gallery_ids': g_ids}, CLIP_FEAT_DIR / 'scores_clip.pt')
print('scores_clip.pt:', tuple(S.shape))

for name in ('val', 'val_zs'):
    ip = CLIP_FEAT_DIR / f'{name}.npz'
    tp = CLIP_FEAT_DIR / f'{name}_text.npz'
    if not (ip.exists() and tp.exists()): continue
    zi = np.load(ip); zt = np.load(tp)
    Sv = _score(zt['embeddings'], zi['embeddings'])
    torch.save({'scores': Sv, 'query_ids': zt['ids'], 'gallery_ids': zi['ids']},
               CLIP_FEAT_DIR / f'scores_{name}.pt')
    print(f'scores_{name}.pt:', tuple(Sv.shape))

for f in CLIP_FEAT_DIR.rglob('*'):
    if f.is_file():
        sync_chunk_to_drive(f, LOCAL_ROOT, DRIVE_OUTPUT_ROOT, background=True)
wait_for_pending_syncs()
print('CLIP inference done.')